# BDL Capacity Planning - Exploratory Data Analysis (EDA)
**Bharat Dynamics Limited (BDL) - Ministry of Defence, Government of India**

This notebook conducts the Exploratory Data Analysis (EDA) for BDL's capacity planning dataset, which models production lines for guided missiles, anti-tank guided missiles (ATGMs), torpedoes, and countermeasures. We analyze distributions, bottlenecks, and the capacity load urgency curve.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']

# Load dataset
df = pd.read_csv('data/bdl_production_planning_data.csv')
print(f"Dataset shape: {df.shape}")

## 1. Distribution of Operation Time and Total Standard Minute Hours (SMH) by Weapon System

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
weapon_order = sorted(df['weapon_system'].unique())

sns.boxplot(ax=axes[0], data=df, x='weapon_system', y='operation_time_min', order=weapon_order, palette='Set2')
axes[0].set_title('Operation Time (minutes) by Weapon System', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Operation Time (min)')

sns.boxplot(ax=axes[1], data=df, x='weapon_system', y='total_smh', order=weapon_order, palette='Set2')
axes[1].set_title('Total Standard Minute Hours (SMH) by Weapon System', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Total SMH')

plt.tight_layout()
plt.show()

## 2. Heatmap of Utilization Rate by Work Center vs Weapon System

In [ ]:
pivot_df = df.pivot_table(values='utilization_rate', index='work_center_code', columns='weapon_system', aggfunc='mean')
wc_sequence = ['WC_MACH', 'WC_SHEET', 'WC_SMT', 'WC_ELEC', 'WC_SEEKER', 'WC_PROP', 'WC_WARHEAD', 
               'WC_TORPEDO', 'WC_INTEGRATION', 'WC_LAUNCHER', 'WC_PROOF', 'WC_QA_INSP', 'WC_PACK']
wc_sequence = [wc for wc in wc_sequence if wc in pivot_df.index]
pivot_df = pivot_df.reindex(wc_sequence)

plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df, annot=True, fmt=".2f", cmap="YlOrRd", cbar_kws={'label': 'Average Utilization Rate'})
plt.title('Average Work Center Utilization Rate by Weapon System', fontsize=14, fontweight='bold')
plt.ylabel('Work Center Code')
plt.xlabel('Weapon System')
plt.tight_layout()
plt.show()

## 3. Average Bottleneck Rate per Work Center

In [ ]:
wc_bottlenecks = df.groupby('work_center_code')['bottleneck_flag'].mean().reset_index()
wc_bottlenecks['seq'] = wc_bottlenecks['work_center_code'].map(lambda x: wc_sequence.index(x) if x in wc_sequence else 99)
wc_bottlenecks = wc_bottlenecks.sort_values('seq')

plt.figure(figsize=(12, 6))
sns.barplot(data=wc_bottlenecks, x='work_center_code', y='bottleneck_flag', palette='Reds_d')
plt.title('Average Bottleneck Rate per Work Center (utilization > 85% & critical path)', fontsize=14, fontweight='bold')
plt.xlabel('Work Center Code')
plt.ylabel('Bottleneck Probability')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Urgency Curve: Capacity Load (Utilization) vs Days to Delivery

In [ ]:
df['delivery_days_bin'] = (df['contracted_delivery_days'] // 30) * 30
urgency_df = df.groupby('delivery_days_bin')['utilization_rate'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=urgency_df, x='delivery_days_bin', y='utilization_rate', marker='o', color='crimson', linewidth=2.5)
plt.axhline(y=0.85, color='orange', linestyle='--', label='Warning Threshold (85%)')
plt.axhline(y=1.0, color='red', linestyle='--', label='Overload Threshold (100%)')
plt.title('Average Capacity Load (Utilization) vs Days to Delivery (Urgency Curve)', fontsize=14, fontweight='bold')
plt.xlabel('Contracted Days to Delivery (30-day Bins)')
plt.ylabel('Average Utilization Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Rework Rate by Process Sheet Type and Sub-Assembly Stage

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

sns.boxplot(ax=axes[0], data=df, x='process_sheet_type', y='rework_rate_pct', palette='Pastel1')
axes[0].set_title('Rework Rate (%) by Process Sheet Type', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Process Sheet Type')
axes[0].set_ylabel('Rework Rate (%)')

sns.boxplot(ax=axes[1], data=df, x='sub_assembly_stage', y='rework_rate_pct', palette='Pastel2')
axes[1].set_title('Rework Rate (%) by Sub-Assembly Stage', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sub-Assembly Stage')
axes[1].set_ylabel('Rework Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()